# Step 0 + Step 1: Compute Check & Dataset Inspection

**Project:** Sickle Cell Detection - CNN research/portfolio tool

**Important framing:** this notebook is part of an **experimental research/screening
tool**, not a medical diagnostic device. Nothing produced here should be interpreted
as a diagnosis or a substitute for laboratory testing or a medical professional.

**What this notebook does:**
1. Confirms we actually have a usable GPU here in Colab (Step 0).
2. Downloads the two datasets into clearly separated folders (never mixed).
3. Inspects their structure, labels, image counts, and license/documentation info
   so we can decide - *before* writing any training code - whether they're
   scientifically usable and how they should be split (Step 1).

**What you should do:** Run the cells top to bottom (`Runtime > Run all`, or one
by one with Shift+Enter). Then copy the printed output back to Claude in the
main conversation so it can review it with you before we build the full pipeline.
Nothing in this notebook trains a model or makes irreversible changes - it only
downloads data and prints information about it.


## 1. Confirm the GPU

**Why:** training two CNNs with cross-validation is only realistic with a GPU.
Colab's free tier *usually* gives you one, but not always (it depends on
availability and your usage quota), so we check rather than assume.

If this prints `No GPU found`, go to `Runtime > Change runtime type` and pick a
GPU (e.g. T4), then re-run this cell before continuing.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv 2>/dev/null || echo "No GPU found by nvidia-smi"


## 2. Get the project code and install dependencies

**Why:** we keep one `requirements.txt` in the GitHub repo so the exact same
package versions are used here as were planned on the development machine -
this avoids "it worked on my machine" version-mismatch bugs later.


In [ ]:
# Clone the project repo (this branch) so requirements.txt and later scripts are available
import os
REPO_URL = "https://github.com/aural0i/Sickle-cell-detection"
BRANCH = "claude/sickle-cell-cnn-research-g6fipc"
if not os.path.isdir("/content/Sickle-cell-detection"):
    !git clone --branch {BRANCH} {REPO_URL} /content/Sickle-cell-detection
%cd /content/Sickle-cell-detection


In [ ]:
# Colab already ships a GPU-matched torch/torchvision preinstalled - forcing a
# specific pinned version here commonly fails (it did: torchvision==0.20.1
# wasn't installable) or, worse, can silently break GPU support. So we skip
# reinstalling those two and install everything else from requirements.txt.
with open("requirements.txt") as f:
    lines = [l for l in f if not l.strip().startswith(("torch==", "torch>=", "torchvision=="))]
with open("/tmp/requirements_colab.txt", "w") as f:
    f.writelines(lines)

!pip install -q -r /tmp/requirements_colab.txt

import torch, torchvision
print("Using Colab's preinstalled torch:", torch.__version__)
print("Using Colab's preinstalled torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Kaggle API credentials

**Why:** the Kaggle API needs a personal credential tied to your Kaggle account
to download datasets. Kaggle now issues this as a single **API token** (the
kind starting with `KGAT_`) instead of the older downloadable `kaggle.json`
file - the code below uses that new token method.

**How to get it (one-time, per Colab session):**
1. Go to https://www.kaggle.com/settings/api
2. Click "Create New Token" (or similar) to generate a token starting with
   `KGAT_`. Copy it - Kaggle only shows it once.
3. Run the cell below. It will prompt you to **paste the token** using a
   hidden input box (like a password field), so it is never printed on
   screen, saved in this notebook's output, or committed to the repo.

If you generated the older-style `kaggle.json` instead (Kaggle still offers
this under "Legacy API Credentials" on that same page), tell Claude and it
will give you an alternate version of this cell that uses the file-upload
method instead.


In [ ]:
import os
from getpass import getpass

# Paste your KGAT_... token when prompted. It is entered via a hidden field
# and stored only in this Colab runtime's memory (os.environ), not written to
# any file, not printed, and not committed to the repo.
os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token (KGAT_...): ")

import kaggle
kaggle.api.authenticate()
print("Kaggle credentials accepted.")


## 4. Download the primary dataset (Kaggle)

**Dataset:** Sickle Cell Disease Dataset (Tushabe et al.)
**Target folder:** `data/train_source/` (kept completely separate from the
external validation data - the task requires we never mix the two).

We also pull the dataset's own metadata (including its stated license) so we
can check the license terms before deciding this is usable in a public
portfolio project.


In [ ]:
import kaggle

os.makedirs("data/train_source", exist_ok=True)
kaggle.api.authenticate()

# Pull dataset metadata (includes license info) without downloading files yet
kaggle.api.dataset_metadata("florencetushabe/sickle-cell-disease-dataset", path=".")
with open("dataset-metadata.json") as f:
    print(f.read())


In [ ]:
# Now download and unzip the actual dataset files
kaggle.api.dataset_download_files(
    "florencetushabe/sickle-cell-disease-dataset",
    path="data/train_source",
    unzip=True,
)
print("Download complete.")


## 5. Download the external validation dataset (Zenodo)

**Dataset:** erythrocytesIDB - https://zenodo.org/records/18299474
**Target folder:** `data/external_val/`

We use Zenodo's own API to list the exact files attached to this record
(including their license) rather than guessing a filename, since Claude
could not view this page directly (network policy in the dev session blocks
Zenodo) and has not seen this record's contents yet.


In [ ]:
import requests

RECORD_ID = "18299474"
r = requests.get(f"https://zenodo.org/api/records/{RECORD_ID}")
r.raise_for_status()
record = r.json()

print("Title:", record["metadata"].get("title"))
print("License:", record["metadata"].get("license"))
print("Access right:", record["metadata"].get("access_right"))
print("Creators:", record["metadata"].get("creators"))
print()
print("Description:")
print(record["metadata"].get("description"))
print()
print("Files in this record:")
for f in record["files"]:
    print(f"  {f['key']}  ({f['size']/1e6:.1f} MB)")


In [ ]:
import os

os.makedirs("data/external_val", exist_ok=True)

for f in record["files"]:
    url = f["links"]["self"]
    out_path = os.path.join("data/external_val", f["key"])
    print(f"Downloading {f['key']} ...")
    resp = requests.get(url, stream=True)
    resp.raise_for_status()
    with open(out_path, "wb") as out:
        for chunk in resp.iter_content(chunk_size=8192):
            out.write(chunk)

print("Download complete. Files saved to data/external_val/")


In [ ]:
# If any downloaded file is a zip/archive, extract it in place
import zipfile, tarfile

for fname in os.listdir("data/external_val"):
    fpath = os.path.join("data/external_val", fname)
    if fname.lower().endswith(".zip"):
        print("Extracting", fname)
        with zipfile.ZipFile(fpath) as z:
            z.extractall("data/external_val")
    elif fname.lower().endswith((".tar", ".tar.gz", ".tgz")):
        print("Extracting", fname)
        with tarfile.open(fpath) as t:
            t.extractall("data/external_val")

print("Done. Current contents of data/external_val:")
print(os.listdir("data/external_val"))


## 6. Inspect folder structure, classes, and image counts

**Why:** before we design a train/val/test split or a leakage-prevention
strategy, we need to actually see how these datasets are organized - are
images grouped by patient/slide/sample? What are the class folder names?
Are there README/LICENSE files bundled inside the download itself?

This cell just *looks*, it doesn't change anything.


In [ ]:
def describe_tree(root, max_depth=3, max_files_per_dir=5):
    root = os.path.abspath(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(dirpath) or dirpath}/  ({len(filenames)} files, {len(dirnames)} subfolders)")
        for fn in sorted(filenames)[:max_files_per_dir]:
            print(f"{indent}  - {fn}")
        if len(filenames) > max_files_per_dir:
            print(f"{indent}  ... ({len(filenames) - max_files_per_dir} more files)")

print("=" * 60)
print("PRIMARY DATASET: data/train_source/")
print("=" * 60)
describe_tree("data/train_source")


In [ ]:
print("=" * 60)
print("EXTERNAL DATASET: data/external_val/")
print("=" * 60)
describe_tree("data/external_val")


In [ ]:
# Look for any bundled documentation/license files inside the downloads themselves
import glob

print("Documentation/license-like files found inside data/train_source/:")
for pattern in ["*README*", "*readme*", "*LICENSE*", "*license*", "*.txt", "*.md", "*.pdf", "*.csv"]:
    for p in glob.glob(f"data/train_source/**/{pattern}", recursive=True):
        print(" ", p)

print()
print("Documentation/license-like files found inside data/external_val/:")
for pattern in ["*README*", "*readme*", "*LICENSE*", "*license*", "*.txt", "*.md", "*.pdf", "*.csv"]:
    for p in glob.glob(f"data/external_val/**/{pattern}", recursive=True):
        print(" ", p)


In [ ]:
# Print the contents of any small text/markdown documentation files found, so we can read them here
for pattern in ["data/train_source/**/*README*", "data/train_source/**/*readme*",
                 "data/train_source/**/*LICENSE*", "data/train_source/**/*license*",
                 "data/external_val/**/*README*", "data/external_val/**/*readme*",
                 "data/external_val/**/*LICENSE*", "data/external_val/**/*license*"]:
    for p in glob.glob(pattern, recursive=True):
        try:
            size = os.path.getsize(p)
            if size < 20000:  # only print small text files
                print("=" * 60)
                print(p)
                print("=" * 60)
                with open(p, "r", errors="replace") as fh:
                    print(fh.read())
                print()
        except Exception as e:
            print(f"Could not read {p}: {e}")


In [ ]:
# Count images per top-level class folder, image formats, and image dimensions (sampled)
from PIL import Image
from collections import Counter

def summarize_images(root, label=""):
    print(f"--- Image summary for {label or root} ---")
    ext_counter = Counter()
    class_counter = Counter()
    dims = Counter()
    modes = Counter()
    n_examined = 0
    sample_paths = []

    for dirpath, dirnames, filenames in os.walk(root):
        img_files = [f for f in filenames if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))]
        if not img_files:
            continue
        rel_class = os.path.relpath(dirpath, root)
        class_counter[rel_class] += len(img_files)
        for f in img_files:
            ext_counter[os.path.splitext(f)[1].lower()] += 1
            full = os.path.join(dirpath, f)
            if n_examined < 300:  # sample up to 300 images for dimension/mode check (keep this cell fast)
                try:
                    with Image.open(full) as im:
                        dims[im.size] += 1
                        modes[im.mode] += 1
                except Exception as e:
                    print(f"  Could not open {full}: {e}")
                n_examined += 1
            if len(sample_paths) < 5:
                sample_paths.append(full)

    print("Images per folder (treat as candidate classes):")
    for k, v in sorted(class_counter.items(), key=lambda x: -x[1]):
        print(f"  {k}: {v} images")
    print("File extensions found:", dict(ext_counter))
    print(f"Image dimensions (sampled {n_examined} images):", dict(dims.most_common(10)))
    print("Color modes (sampled):", dict(modes))
    print("Example file paths:")
    for p in sample_paths:
        print(" ", p)
    print()

summarize_images("data/train_source", "PRIMARY (train_source)")
summarize_images("data/external_val", "EXTERNAL (external_val)")


## 8. Investigate a possible healthy-comparison dataset (Acevedo et al., Mendeley)

**Why we're looking at this:** erythrocytesIDB (Section 5 above) turned out not
to have a real healthy/normal-donor comparison group - every image in it comes
from Sickle Cell Disease patients. Without a healthy comparison group, external
validation can't properly test the negative/specificity side of the task. You
suggested checking the Acevedo et al. Barcelona peripheral blood cell dataset
(https://data.mendeley.com/datasets/snkd93bnjr/1) as a possible source of
isolated normal red blood cell images.

**What Claude could NOT verify:** this dataset's page (`data.mendeley.com`) is
blocked by this development session's network policy, same as Kaggle and
Zenodo were. From memory, Claude recalls this dataset's labeled classes are
mostly **white blood cell types** (neutrophils, eosinophils, basophils,
lymphocytes, monocytes, immature granulocytes) plus platelets plus
**erythroblasts** (immature, nucleated red cell precursors - not mature
biconcave red blood cells). If that's right, it may not actually contain a
dedicated "normal mature erythrocyte" class. **This is an unverified
recollection, not a confirmed fact** - the cells below try to check the real
thing using Colab's own internet access, since this notebook runs somewhere
that isn't network-restricted.

This is a bigger download than the other two (this dataset is roughly on the
order of 1-2 GB total) - flagging that before you run it, since it'll take a
few minutes and use some of your Colab session's disk/bandwidth allowance.

We save anything from this dataset into its own folder, `data/healthy_reference/`
- kept just as separate from `train_source/` and `external_val/` as those two
are from each other, since it would be a third dataset with its own license
and provenance that must never be silently mixed with the others.


In [ ]:
import requests

MENDELEY_URL = "https://data.mendeley.com/datasets/snkd93bnjr/1"

# Try to fetch the page directly. This may or may not work depending on how
# the site is built (client-side rendered pages sometimes don't expose much
# in the raw HTML) - we handle that gracefully below rather than assuming.
headers = {"User-Agent": "Mozilla/5.0 (compatible; research-notebook/1.0)"}
resp = requests.get(MENDELEY_URL, headers=headers, timeout=30)
print("HTTP status:", resp.status_code)
html = resp.text
print("Page size:", len(html), "characters")

import re, json as _json

# Many modern (Next.js-based) sites embed a JSON blob with full page data in
# a <script id="__NEXT_DATA__"> tag. Try to find and parse it - if the schema
# doesn't match what we expect, we still print raw snippets so nothing is lost.
match = re.search(
    r'<script id="__NEXT_DATA__"[^>]*>(.*?)</script>', html, re.DOTALL
)

def find_keys(obj, wanted_substrings, path="", results=None):
    if results is None:
        results = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            newpath = f"{path}.{k}" if path else k
            if any(w in k.lower() for w in wanted_substrings):
                if isinstance(v, (str, int, float, bool)) or v is None:
                    results.append((newpath, v))
            find_keys(v, wanted_substrings, newpath, results)
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            find_keys(v, wanted_substrings, f"{path}[{i}]", results)
    return results

if match:
    try:
        data = _json.loads(match.group(1))
        interesting = find_keys(
            data, ["title", "name", "license", "file", "download", "size", "description", "doi"]
        )
        print(f"Found __NEXT_DATA__ JSON with {len(interesting)} interesting fields. Showing up to 60:")
        for path, val in interesting[:60]:
            val_str = str(val)
            if len(val_str) > 200:
                val_str = val_str[:200] + "..."
            print(f"  {path}: {val_str}")
    except Exception as e:
        print("Found __NEXT_DATA__ but could not parse it as JSON:", e)
        print("Raw snippet:", match.group(1)[:2000])
else:
    print("No __NEXT_DATA__ script tag found. Printing a raw HTML snippet instead")
    print("so we can see what's actually on the page:")
    print(html[:3000])


### If the automated check above didn't give a clean file list

Some dataset sites render their download links client-side (via JavaScript
after the page loads), which a plain `requests.get()` can't see - so the cell
above may come back inconclusive rather than broken. If that happens, do this
instead:

1. Open https://data.mendeley.com/datasets/snkd93bnjr/1 in your own browser.
2. Check the **class list and file counts shown on the page** - specifically,
   does it list a category for plain/mature red blood cells (erythrocytes),
   separate from white blood cell types, platelets, and erythroblasts? Note
   the exact class names and counts shown.
3. Check the **license** shown on the page.
4. Report both back to Claude before downloading anything - if it turns out
   there's no isolated mature-RBC class, downloading ~1-2 GB would be wasted
   effort, so let's confirm first.
5. If it does look right, click "Download All Files", then either:
   - Upload the resulting archive into Colab via the file browser (left
     sidebar > folder icon > upload), or
   - Upload it to your Google Drive and mount Drive in Colab
   and tell Claude which you did so the next notebook cell can be written to
   find it in the right place.


## 9. Download and inspect a second healthy-comparison candidate: Chula-RBC-12-Dataset

The Acevedo/Mendeley check in Section 8 was blocked by that site's own
anti-bot protection (Cloudflare), not by network policy - a dead end we're
leaving documented above rather than deleting.

You found a better-looking candidate instead: **Chula-RBC-12-Dataset**
(Naruenatthanaset et al., 2021), hosted on Zenodo
(https://zenodo.org/records/5638201). Per the dataset's own documentation:
706 whole blood-smear images (640x480) with over 20,000 individually
labeled red blood cells across 12 shape classes, where **class 0 is
explicitly "Normal cell"** - a real healthy-comparison label, not a proxy.
Small download (~58 MB), so this is a quick one.

**License - needs your eyes too:** the dataset's GitHub repo
(Chula-PIC-Lab/Chula-RBC-12-Dataset) has an MIT License file, which is
permissive (use/modify/redistribute, just keep the copyright+license
notice). But Zenodo lists the *dataset's own* license separately as
`"Other (Open)"`, and Claude could not load the Zenodo page itself to see
if it points to the same MIT terms or something else - `zenodo.org` is
blocked from Claude's development session, the same as before. **Please
check the "License" section on the Zenodo page yourself** and report back
exactly what it says, so we don't assume MIT covers the images just because
it covers the code.

Also note: **citation is required** if this dataset is used - Naruenatthanaset
et al., "Red Blood Cell Segmentation with Overlapping Cell Separation and
Classification on Imbalanced Dataset," arXiv:2012.01321 (2021). We'll credit
this properly in the project's README/docs regardless of what else we decide.

**This section only downloads and inspects the label format - it does not
extract or use the "Normal cell" annotations yet.** That's a deliberate
pause point, since these are whole-smear images with per-cell coordinate
labels rather than pre-cropped single-cell images like our other datasets -
we need to actually see the label file format before writing extraction
code, rather than guessing at it.


In [ ]:
import requests, os

os.makedirs("data/healthy_comparison", exist_ok=True)

CHULA_URL = "https://zenodo.org/records/5638201/files/Chula-PIC-Lab/Chula-RBC-12-Dataset-dataset.zip?download=1"
out_path = "data/healthy_comparison/Chula-RBC-12-Dataset-dataset.zip"

print("Downloading Chula-RBC-12-Dataset (~58 MB)...")
resp = requests.get(CHULA_URL, stream=True, timeout=60)
resp.raise_for_status()
with open(out_path, "wb") as out:
    for chunk in resp.iter_content(chunk_size=8192):
        out.write(chunk)
print("Download complete:", os.path.getsize(out_path) / 1e6, "MB")


In [ ]:
import zipfile

with zipfile.ZipFile(out_path) as z:
    z.extractall("data/healthy_comparison")

print("Extracted. Top-level contents of data/healthy_comparison/:")
print(os.listdir("data/healthy_comparison"))


In [ ]:
# Folder structure (reuses the describe_tree function defined in Section 6 above -
# run that section first if you haven't, otherwise re-run this cell after defining it)
print("=" * 60)
print("HEALTHY-COMPARISON CANDIDATE: data/healthy_comparison/")
print("=" * 60)
describe_tree("data/healthy_comparison")


In [ ]:
# What file types actually exist here? This tells us whether labels are one
# file per image (many small .txt/.xml/.json files) or one master file.
from collections import Counter

ext_counter = Counter()
for dirpath, dirnames, filenames in os.walk("data/healthy_comparison"):
    for f in filenames:
        ext_counter[os.path.splitext(f)[1].lower()] += 1

print("File extensions found:")
for ext, count in ext_counter.most_common():
    print(f"  {ext or '(no extension)'}: {count}")


In [ ]:
# Print the full contents of a few non-image files (likely the label/annotation
# files) so we can see the exact format the README didn't fully specify.
import glob

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
non_image_exts = {ext for ext in ext_counter if ext and ext not in image_exts}
print("Non-image file extensions:", non_image_exts)
print()

shown = 0
for dirpath, dirnames, filenames in os.walk("data/healthy_comparison"):
    for f in sorted(filenames):
        ext = os.path.splitext(f)[1].lower()
        if ext in non_image_exts and shown < 5:
            full = os.path.join(dirpath, f)
            size = os.path.getsize(full)
            print("=" * 60)
            print(full, f"({size} bytes)")
            print("=" * 60)
            if size < 5000:
                with open(full, "r", errors="replace") as fh:
                    print(fh.read())
            else:
                print("(file is large - showing first 1000 characters)")
                with open(full, "r", errors="replace") as fh:
                    print(fh.read(1000))
            print()
            shown += 1


In [ ]:
# Do image files and label files share a naming pattern? (e.g. 001.jpg + 001.txt)
img_files = sorted(glob.glob("data/healthy_comparison/**/*.jpg", recursive=True) +
                    glob.glob("data/healthy_comparison/**/*.png", recursive=True))
print(f"Total image files found: {len(img_files)}")
print()
print("First 5 images and any same-named files next to them:")
for p in img_files[:5]:
    base = os.path.splitext(p)[0]
    matches = [m for m in glob.glob(base + ".*") if m != p]
    print(" ", p, "-> other files with same name:", matches)


In [ ]:
# Look for any bundled README/LICENSE/citation files inside the download itself
print("Documentation/license-like files found inside data/healthy_comparison/:")
for pattern in ["*README*", "*readme*", "*LICENSE*", "*license*", "*CITATION*", "*.md"]:
    for p in glob.glob(f"data/healthy_comparison/**/{pattern}", recursive=True):
        print(" ", p)


### Report back before we go further

Per your instructions, this is a pause point - copy back:
- The file-extension counts and the printed label-file contents above (so we
  can nail down the exact annotation format and how to pull out just the
  "Normal cell" / class-0 entries)
- The image/label naming-correspondence check
- Whatever the Zenodo page's "License" section actually says, in full

Once we can see the real label format, Claude will write the extraction code
to pull out just the Normal-cell crops/coordinates, count how many there
are, and report back whether it's a large enough sample before anything gets
wired into the evaluation pipeline.


## 10. What to do with this output

Copy everything printed above (or share this notebook) back to Claude in the
main conversation. Claude will use it to:

- Confirm whether the primary dataset is genuinely microscopy imagery suitable
  for this task, and show you the class definitions/counts/example structure
- Determine whether the external dataset (erythrocytesIDB) is scientifically
  compatible enough for external validation, or whether it should be used
  differently (or not at all) - it will **not** force an invalid comparison
- Check whether the Acevedo et al. dataset (Section 8) actually contains an
  isolated normal-erythrocyte class in usable numbers, to close the
  healthy-comparison gap in external validation
- Review the license/usage terms captured above and flag any restriction
  (non-commercial-only, attribution requirements, etc.) before this is used in
  a public portfolio
- Check for patient/slide/sample grouping information needed to prevent data
  leakage when we split the data later

**Nothing has been trained or split yet** - this notebook only downloaded and
described the data.
